# Statistical Analysis with SciRS2

This notebook demonstrates the statistical computing capabilities of
`scirs2-stats`, which mirrors `scipy.stats`. SciRS2 provides 100+
probability distributions, hypothesis tests, correlation measures,
and regression models.

Topics covered:
- Probability distributions (Normal, Beta, Gamma, Poisson)
- PDF and CDF evaluation
- Random sampling
- Hypothesis testing (t-test, normality tests)
- Correlation analysis
- Linear regression

In [ ]:
:dep scirs2-core = { path = "../scirs2-core" }
:dep scirs2-stats = { path = "../scirs2-stats" }

In [ ]:
use scirs2_core::ndarray::{array, Array1};
use scirs2_stats::Distribution;

## Probability Distributions

SciRS2 provides a unified `Distribution` trait with `pdf`, `cdf`, `mean`,
`var`, and `rvs` methods. Distributions are created through factory functions
in `scirs2_stats::distributions`.

In [ ]:
use scirs2_stats::distributions;

// Normal distribution: N(mu=0, sigma=1)
let normal = distributions::norm(0.0f64, 1.0).expect("norm failed");

println!("=== Standard Normal Distribution N(0, 1) ===");
println!("  Mean:     {:.4}", normal.mean());
println!("  Variance: {:.4}", normal.var());
println!("  PDF(0):   {:.6} (expected: 0.398942)", normal.pdf(0.0));
println!("  CDF(0):   {:.6} (expected: 0.500000)", normal.cdf(0.0));
println!("  CDF(1.96):{:.6} (expected: 0.975002)", normal.cdf(1.96));
println!("  CDF(-1.96):{:.6} (expected: 0.024998)", normal.cdf(-1.96));

In [ ]:
// Beta distribution: Beta(alpha=2, beta=5)
// Useful for modeling proportions and probabilities
let beta_dist = distributions::beta(2.0f64, 5.0, 0.0, 1.0).expect("beta failed");

println!("=== Beta Distribution Beta(2, 5) ===");
println!("  Mean:     {:.4} (expected: 0.2857)", beta_dist.mean());
println!("  Variance: {:.4} (expected: 0.0255)", beta_dist.var());

// Evaluate PDF at several points
for &x in &[0.1, 0.2, 0.3, 0.5, 0.8] {
    println!("  PDF({:.1}): {:.6}", x, beta_dist.pdf(x));
}

In [ ]:
// Gamma distribution: Gamma(shape=3, scale=2)
// Common in survival analysis and queueing theory
let gamma_dist = distributions::gamma(3.0f64, 2.0, 0.0).expect("gamma failed");

println!("=== Gamma Distribution Gamma(3, 2) ===");
println!("  Mean:     {:.4} (expected: 6.0)", gamma_dist.mean());
println!("  Variance: {:.4} (expected: 12.0)", gamma_dist.var());
println!("  PDF(4):   {:.6}", gamma_dist.pdf(4.0));
println!("  CDF(6):   {:.6}", gamma_dist.cdf(6.0));

In [ ]:
// Poisson distribution: Poisson(lambda=5)
// Models count data (number of events in fixed interval)
let poisson = distributions::poisson(5.0f64, 0.0).expect("poisson failed");

println!("=== Poisson Distribution Poisson(5) ===");
println!("  Mean:     {:.4}", poisson.mean());
println!("  Variance: {:.4}", poisson.var());

// PMF for k = 0, 1, ..., 10
println!("  PMF values:");
for k in 0..=10 {
    let p = poisson.pmf(k as f64);
    let bar = "#".repeat((p * 50.0) as usize);
    println!("    P(X={:2}) = {:.4}  {}", k, p, bar);
}

## Random Sampling

All distributions support random variate generation via the `rvs` method.
We can use this to verify distribution properties empirically.

In [ ]:
use scirs2_stats::{mean, var};

// Draw 10000 samples from N(5, 2^2)
let normal_52 = distributions::norm(5.0f64, 2.0).expect("norm failed");
let samples = normal_52.rvs(10000).expect("rvs failed");
let samples_arr = Array1::from_vec(samples);

let empirical_mean = mean(&samples_arr.view()).expect("mean failed");
let empirical_var = var(&samples_arr.view(), 1, None).expect("var failed");

println!("N(5, 4) with 10000 samples:");
println!("  Theoretical mean: 5.0000");
println!("  Empirical mean:   {:.4}", empirical_mean);
println!("  Theoretical var:  4.0000");
println!("  Empirical var:    {:.4}", empirical_var);

## Hypothesis Testing

### One-Sample t-test

Tests whether the mean of a sample differs significantly from a
hypothesized population mean.

In [ ]:
use scirs2_stats::ttest_1samp;
use scirs2_stats::tests::ttest::Alternative;

// Suppose we measure reaction times (ms) for 15 subjects
let reaction_times = array![
    245.0, 252.0, 238.0, 261.0, 249.0,
    255.0, 243.0, 258.0, 247.0, 251.0,
    260.0, 244.0, 253.0, 248.0, 256.0
];

// H0: mean reaction time = 250 ms
let result = ttest_1samp(&reaction_times.view(), 250.0, Alternative::TwoSided, "propagate")
    .expect("ttest_1samp failed");

println!("One-sample t-test: H0: mu = 250 ms");
println!("  t-statistic: {:.4}", result.statistic);
println!("  p-value:     {:.4}", result.pvalue);
if result.pvalue < 0.05 {
    println!("  Result: Reject H0 at alpha=0.05");
} else {
    println!("  Result: Fail to reject H0 at alpha=0.05");
}

### Two-Sample t-test

Compares means between two independent groups.

In [ ]:
use scirs2_stats::ttest_ind;

// Treatment group vs control group
let treatment = array![78.0, 82.0, 85.0, 79.0, 88.0, 91.0, 84.0, 86.0];
let control   = array![72.0, 75.0, 78.0, 71.0, 74.0, 77.0, 73.0, 76.0];

let treatment_mean = mean(&treatment.view()).expect("mean failed");
let control_mean = mean(&control.view()).expect("mean failed");

// Two-sample t-test assuming equal variances
let result = ttest_ind(
    &treatment.view(), &control.view(),
    true,  // equal_var
    Alternative::TwoSided,
    "propagate"
).expect("ttest_ind failed");

println!("Two-sample t-test:");
println!("  Treatment mean: {:.2}", treatment_mean);
println!("  Control mean:   {:.2}", control_mean);
println!("  t-statistic:    {:.4}", result.statistic);
println!("  p-value:        {:.6}", result.pvalue);
if result.pvalue < 0.05 {
    println!("  Result: Significant difference at alpha=0.05");
} else {
    println!("  Result: No significant difference at alpha=0.05");
}

### Normality Testing

Before applying parametric tests, we should verify that data is
approximately normally distributed.

In [ ]:
use scirs2_stats::{shapiro, anderson_darling};

let data = array![
    5.1, 4.9, 6.2, 5.7, 5.5, 5.1, 5.2, 5.0, 5.3, 5.4,
    5.6, 5.8, 5.9, 6.0, 5.2, 5.4, 5.3, 5.1, 5.2, 5.0
];

// Shapiro-Wilk test
let (w, p_sw) = shapiro(&data.view()).expect("shapiro failed");
println!("Shapiro-Wilk test:");
println!("  W = {:.4}, p = {:.4}", w, p_sw);

// Anderson-Darling test
let (a2, p_ad) = anderson_darling(&data.view()).expect("anderson_darling failed");
println!("\nAnderson-Darling test:");
println!("  A2 = {:.4}, p = {:.4}", a2, p_ad);

if p_sw > 0.05 {
    println!("\nConclusion: Data appears normally distributed (fail to reject normality)");
} else {
    println!("\nConclusion: Data may not be normally distributed");
}

## Correlation Analysis

Correlation measures the strength and direction of linear (Pearson)
or monotonic (Spearman) relationships between variables.

In [ ]:
use scirs2_stats::{pearsonr, spearmanr};

// Strong positive linear relationship
let x = array![1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 10.0];
let y = array![2.1, 4.0, 6.2, 7.8, 10.1, 12.0, 13.9, 16.1, 18.0, 20.2];

let (r, p_r) = pearsonr(&x.view(), &y.view(), "two-sided").expect("pearsonr failed");
println!("Pearson correlation:");
println!("  r = {:.6}, p = {:.2e}", r, p_r);

let rho_result = spearmanr(&x.view(), &y.view(), "two-sided").expect("spearmanr failed");
println!("\nSpearman rank correlation:");
println!("  rho = {:.6}", rho_result.0);

## Linear Regression

Ordinary least squares (OLS) regression fits a linear model y = X*beta + epsilon
minimizing the sum of squared residuals.

In [ ]:
use scirs2_stats::regression::linear_regression;

// Predictor and response
let x = array![[1.0], [2.0], [3.0], [4.0], [5.0],
               [6.0], [7.0], [8.0], [9.0], [10.0]];
let y = array![2.3, 4.1, 6.2, 7.8, 10.1, 12.0, 13.9, 16.1, 18.0, 20.2];

let result = linear_regression(&x.view(), &y.view(), None)
    .expect("linear_regression failed");

println!("Linear Regression: y = b0 + b1*x");
println!("  Intercept:    {:.4}", result.intercept);
println!("  Coefficients: {:?}", result.coefficients);
println!("  R-squared:    {:.6}", result.r_squared);

// Predictions
println!("\nPredictions vs Actual:");
for i in 0..y.len() {
    let pred = result.intercept + result.coefficients[0] * x[[i, 0]];
    println!("  x={:.0}: predicted={:.2}, actual={:.1}, residual={:.2}",
        x[[i, 0]], pred, y[i], y[i] - pred);
}

## Non-parametric Tests

When normality assumptions are not met, non-parametric alternatives
like the Mann-Whitney U test can be used.

In [ ]:
use scirs2_stats::mann_whitney;

// Two groups with possibly non-normal distributions
let group_a = array![12.0, 15.0, 18.0, 22.0, 25.0, 28.0, 35.0];
let group_b = array![8.0, 10.0, 13.0, 14.0, 16.0, 19.0, 20.0];

let (u_stat, p_value) = mann_whitney(
    &group_a.view(), &group_b.view(), "two-sided", true
).expect("mann_whitney failed");

println!("Mann-Whitney U test:");
println!("  U-statistic: {:.4}", u_stat);
println!("  p-value:     {:.4}", p_value);
if p_value < 0.05 {
    println!("  Result: Groups differ significantly at alpha=0.05");
} else {
    println!("  Result: No significant difference at alpha=0.05");
}

## Summary

This notebook covered the fundamental statistical tools in SciRS2:

1. **Distributions** -- Normal, Beta, Gamma, Poisson (100+ available)
2. **PDF/CDF** -- Evaluating probability density and cumulative distribution
3. **Random sampling** -- Generating variates and verifying properties
4. **Hypothesis testing** -- t-tests, normality tests, non-parametric tests
5. **Correlation** -- Pearson and Spearman correlation coefficients
6. **Regression** -- Ordinary least squares linear regression

Additional capabilities not covered here:
- Ridge, Lasso, and Elastic Net regression
- Bayesian inference and MCMC sampling
- Survival analysis (Kaplan-Meier, Cox PH)
- Mixture models and KDE
- Conformal prediction and calibration